# EV3 fine-tuning without calibration-set reuse (reviewer point 3)
Re-runs the full fine-tuning experiment so that the **EV3 calibration split (184 images) is never used**:
* early stopping uses an internal, stratified 80/20 split of the **pool** (454 images; near-duplicate groups kept together);
* a second variant uses a **fixed number of epochs** (no model selection at all);
* the **test split (271 images) is evaluated once**, after training.
Inputs: `ev3-frost-grape` and the output of the EV3 notebook (`notebook4873e635e9`: `ev3_results/weapd_<model>.pt`, `ev3_split.csv`).
**GPU T4 x2, Internet On.** Download `ev3_ft_nocal_small.zip`.

In [ ]:
CFG = dict(BACKBONES=["swin_tiny_patch4_window7_224", "convnext_tiny"], INITS=["weapd", "imagenet"], SEEDS=[0, 1, 2],
           EPOCHS=12, LR=1e-4, BATCH=32, INNER_VAL=0.2, R=256, S=224, OUT="/kaggle/working/ev3_ft_nocal")
import os, glob, json, time, numpy as np, pandas as pd, torch, torch.nn.functional as F, timm
from PIL import Image, ImageFile
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
ImageFile.LOAD_TRUNCATED_IMAGES = True; os.makedirs(CFG["OUT"], exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"; print(DEV)

In [ ]:
# 1. Inputs and splits (re-uses the exact EV3 split of the paper)
ann = glob.glob("/kaggle/input/**/_annotations.coco.json", recursive=True); assert ann
EV3_ROOT = os.path.dirname(os.path.dirname(ann[0]))
SPLIT = next(iter(glob.glob("/kaggle/input/**/ev3_split.csv", recursive=True)), None); assert SPLIT, "attach the EV3 notebook output"
CKPT = {b: next(iter(glob.glob(f"/kaggle/input/**/weapd_{b}.pt", recursive=True)), None) for b in CFG["BACKBONES"]}
assert all(CKPT.values()), CKPT
sp = pd.read_csv(SPLIT); sp["k"] = sp.file.str.split("/").str[-2:].str.join("/")
files = {os.path.join(s, f): os.path.join(EV3_ROOT, s, f) for s in ["train", "valid", "test"] for f in os.listdir(os.path.join(EV3_ROOT, s)) if not f.endswith(".json")}
sp["path"] = sp.k.map(files); assert sp.path.notna().all()
pool = sp[sp.split == "pool"].reset_index(drop=True); test = sp[sp.split == "test"].reset_index(drop=True)
grp = pool["group"] if "group" in pool else np.arange(len(pool))
tr_i, va_i = next(StratifiedGroupKFold(int(round(1 / CFG["INNER_VAL"])), shuffle=True, random_state=0).split(pool, pool.y, grp))
print("pool", len(pool), "-> inner train", len(tr_i), "inner val", len(va_i), "| test", len(test), "| calibration split NOT used:", (sp.split == "cal").sum())
json.dump(dict(inner_train=pool.k[tr_i].tolist(), inner_val=pool.k[va_i].tolist(), test=test.k.tolist()), open(os.path.join(CFG["OUT"], "split_used.json"), "w"))

def load(paths):   # uint8 tensors resized to R x R (cached once; augmentation on GPU)
    return torch.stack([torch.from_numpy(np.asarray(Image.open(p).convert("RGB").resize((CFG["R"],) * 2, Image.BICUBIC))).permute(2, 0, 1) for p in paths])
X_pool, X_test = load(pool.path), load(test.path)
Y_pool, Y_test = torch.tensor(pool.y.values), test.y.values
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
def norm(x): return (x.to(DEV).float() / 255 - MEAN) / STD
def aug(x):
    R, S = CFG["R"], CFG["S"]; i, j = np.random.randint(0, R - S + 1, 2); x = x[:, :, i:i + S, j:j + S]
    return torch.flip(x, [3]) if np.random.rand() < 0.5 else x
def center(x): o = (CFG["R"] - CFG["S"]) // 2; return x[:, :, o:o + CFG["S"], o:o + CFG["S"]]
@torch.no_grad()
def predict(m, X):
    m.eval(); out = []
    for i in range(0, len(X), 64):
        x = norm(center(X[i:i + 64]))
        with torch.autocast("cuda", enabled=DEV == "cuda"): out.append(((m(x) + m(torch.flip(x, [3]))) / 2).float().softmax(1)[:, 1].cpu())
    return torch.cat(out).numpy()

In [ ]:
# 2. Fine-tuning: (a) early stopping on the inner pool split, (b) fixed epochs; test evaluated once per run
rows = []
for b in CFG["BACKBONES"]:
    for init in CFG["INITS"]:
        for s in CFG["SEEDS"]:
            torch.manual_seed(s); np.random.seed(s); t0 = time.time()
            if init == "weapd":
                sd = torch.load(CKPT[b], map_location="cpu"); m = timm.create_model(b, pretrained=False, num_classes=11)
                m.load_state_dict(sd); m.reset_classifier(2)
            else:
                m = timm.create_model(b, pretrained=True, num_classes=2)
            m = m.to(DEV); opt = torch.optim.AdamW(m.parameters(), lr=CFG["LR"], weight_decay=0.05)
            steps = CFG["EPOCHS"] * int(np.ceil(len(tr_i) / CFG["BATCH"])); sch = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["LR"], total_steps=steps)
            scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda"); best, best_state, best_ep = -1, None, -1
            for ep in range(CFG["EPOCHS"]):
                m.train(); perm = np.random.permutation(tr_i)
                for i in range(0, len(perm), CFG["BATCH"]):
                    bi = perm[i:i + CFG["BATCH"]]; x = norm(aug(X_pool[bi])); y = Y_pool[bi].to(DEV)
                    opt.zero_grad()
                    with torch.autocast("cuda", enabled=DEV == "cuda"): loss = F.cross_entropy(m(x), y)
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
                pv = predict(m, X_pool[va_i]); au = roc_auc_score(pool.y.values[va_i], pv)
                if au > best: best, best_ep, best_state = au, ep, {k: v.detach().clone() for k, v in m.state_dict().items()}
            p_fixed = predict(m, X_test)                       # (b) last epoch, no selection
            m.load_state_dict(best_state); p_es = predict(m, X_test)   # (a) early stopping on inner pool split
            for var, p in [("early_stop_inner_pool", p_es), ("fixed_epochs", p_fixed)]:
                rows.append(dict(backbone=b, init=init, seed=s, variant=var, inner_val_auroc=round(best, 4), best_epoch=best_ep,
                                 test_auroc=roc_auc_score(Y_test, p), test_bal_acc=balanced_accuracy_score(Y_test, p >= 0.5),
                                 test_macro_f1=f1_score(Y_test, p >= 0.5, average="macro"), minutes=round((time.time() - t0) / 60, 1)))
            print(rows[-2]); print(rows[-1]); del m; torch.cuda.empty_cache()

In [ ]:
# 3. Summary and zip
R_ = pd.DataFrame(rows); R_.to_csv(os.path.join(CFG["OUT"], "ft_nocal_runs.csv"), index=False)
S_ = R_.groupby(["variant", "backbone", "init"])[["test_auroc", "test_bal_acc", "test_macro_f1"]].agg(["mean", "std"]).round(4)
S_.to_csv(os.path.join(CFG["OUT"], "ft_nocal_summary.csv")); print(S_)
import zipfile
zp = "/kaggle/working/ev3_ft_nocal_small.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(CFG["OUT"]): z.write(os.path.join(CFG["OUT"], f), f)
print(zp, "done")